# AI Hardware Cost Trends: 2025–2026
**Source:** Epoch AI — *Data on AI Chip Components*  
**Coverage:** Quarterly cost estimates for major AI accelerators (NVIDIA, AMD, Google, Amazon)  
**Goal:** Show the rise in hardware costs across Logic, CoWoS packaging, HBM memory, and Auxiliary components.

> Costs are **per-chip medians** in USD. Uncertainty bands use the 5th–95th percentile range.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#333355',
    'axes.labelcolor':  '#ccccdd',
    'xtick.color':      '#aaaacc',
    'ytick.color':      '#aaaacc',
    'text.color':       '#eeeeee',
    'grid.color':       '#2a2a4a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.framealpha': 0.3,
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})

QUARTER_ORDER = ['Q1 2024','Q2 2024','Q3 2024','Q4 2024',
                 'Q1 2025','Q2 2025','Q3 2025','Q4 2025','Q1 2026']

DESIGNER_COLORS = {
    'NVIDIA': '#76b900',
    'AMD':    '#ed1c24',
    'Google': '#4285f4',
    'Amazon': '#ff9900',
    'Other':  '#888888',
}

COMPONENT_COLORS = {
    'Logic':     '#4fc3f7',
    'CoWoS':     '#ce93d8',
    'HBM':       '#ef9a9a',
    'Auxiliary': '#a5d6a7',
}

def fmt_billions(x, _):
    if abs(x) >= 1e9:  return f'${x/1e9:.1f}B'
    if abs(x) >= 1e6:  return f'${x/1e6:.0f}M'
    return f'${x:.0f}'

print("✅ Setup complete.")


## 1  Load & Prepare Data

In [ ]:
chip_q   = pd.read_csv('quarterly_by_chip.csv')
design_q = pd.read_csv('quarterly_by_designer.csv')
supply_q = pd.read_csv('supply_denominators.csv')

# Compute total median cost per chip per quarter
cost_cols = {
    'Logic':     'Logic cost (USD) (median)',
    'CoWoS':     'CoWoS cost (USD) (median)',
    'HBM':       'HBM cost (USD) (median)',
    'Auxiliary': 'Auxiliary cost (USD) (median)',
}
for label, col in cost_cols.items():
    chip_q[label] = pd.to_numeric(chip_q[col], errors='coerce').fillna(0)

chip_q['Total Cost'] = chip_q[list(cost_cols.keys())].sum(axis=1)

# Low/high uncertainty bands
chip_q['Total Low']  = (pd.to_numeric(chip_q['Logic cost (USD) (5th percentile)'],  errors='coerce').fillna(0) +
                        pd.to_numeric(chip_q['CoWoS cost (USD) (5th percentile)'],  errors='coerce').fillna(0) +
                        pd.to_numeric(chip_q['HBM cost (USD) (5th percentile)'],    errors='coerce').fillna(0) +
                        pd.to_numeric(chip_q['Auxiliary cost (USD) (5th percentile)'],errors='coerce').fillna(0))
chip_q['Total High'] = (pd.to_numeric(chip_q['Logic cost (USD) (95th percentile)'],  errors='coerce').fillna(0) +
                        pd.to_numeric(chip_q['CoWoS cost (USD) (95th percentile)'],  errors='coerce').fillna(0) +
                        pd.to_numeric(chip_q['HBM cost (USD) (95th percentile)'],    errors='coerce').fillna(0) +
                        pd.to_numeric(chip_q['Auxiliary cost (USD) (95th percentile)'],errors='coerce').fillna(0))

# Ordered quarter axis
chip_q['Q_order'] = pd.Categorical(chip_q['Quarter'], categories=QUARTER_ORDER, ordered=True)

# Clean chip type column
chip_q['Chip type'] = chip_q['Chip type'].fillna('Other')

print(f"Chip rows: {len(chip_q)}  |  Designers: {chip_q['Designer'].unique().tolist()}")
print(f"Quarters:  {sorted(chip_q['Quarter'].unique())}")
chip_q[['Name','Quarter','Designer','Chip type','Total Cost']].head(6)


## 2  Total Quarterly Chip Spend by Designer
Aggregated across all chips each company shipped that quarter.  
Gives a macro view of *who is spending more* as AI hardware scales up.


In [ ]:
# Aggregate by designer + quarter
for label, col in cost_cols.items():
    design_q[label] = pd.to_numeric(design_q[col], errors='coerce').fillna(0)
design_q['Total Cost'] = design_q[list(cost_cols.keys())].sum(axis=1)
design_q['Q_order'] = pd.Categorical(design_q['Quarter'], categories=QUARTER_ORDER, ordered=True)

fig, ax = plt.subplots(figsize=(13, 6))

for designer, grp in design_q.groupby('Designer'):
    grp = grp.sort_values('Q_order')
    color = DESIGNER_COLORS.get(designer, '#aaaaaa')
    ax.plot(grp['Quarter'], grp['Total Cost'], marker='o', linewidth=2.5,
            markersize=7, color=color, label=designer, zorder=3)
    ax.fill_between(grp['Quarter'],
                    pd.to_numeric(design_q.loc[design_q['Designer']==designer,
                                               'Logic cost (USD) (5th percentile)'].fillna(0)),
                    pd.to_numeric(design_q.loc[design_q['Designer']==designer,
                                               'Logic cost (USD) (95th percentile)'].fillna(0)),
                    color=color, alpha=0.07)

ax.set_title('Total AI Chip Component Cost by Designer (2024–2026)', pad=14)
ax.set_xlabel('Quarter')
ax.set_ylabel('Total Quarterly Cost (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_billions))
ax.tick_params(axis='x', rotation=30)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('chart1_designer_trend.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved.")


## 3  Per-Chip Cost Trends: Key AI Accelerators
Individual chip cost trajectories — with uncertainty bands — for the most significant accelerators.


In [ ]:
# Focus on flagship chips with data across multiple quarters
focus_types = ['H100/H200', 'B200', 'B300', 'MI300X', 'MI350X', 'TPU v5p', 'TPU v7', 'Trainium2']

df_focus = chip_q[chip_q['Chip type'].isin(focus_types)].copy()
df_focus = df_focus.sort_values('Q_order')

# Use one row per (chip_type, quarter) — take median if dupes
df_focus = df_focus.groupby(['Chip type', 'Quarter', 'Q_order'], observed=True).agg(
    Total=('Total Cost','median'),
    Low=('Total Low','median'),
    High=('Total High','median'),
    Designer=('Designer','first'),
).reset_index()

fig, ax = plt.subplots(figsize=(14, 7))

for chip_type, grp in df_focus.groupby('Chip type'):
    grp = grp.sort_values('Q_order')
    designer = grp['Designer'].iloc[0]
    color = DESIGNER_COLORS.get(designer, '#aaaaaa')
    ax.plot(grp['Quarter'], grp['Total'], marker='o', linewidth=2,
            markersize=6, color=color, label=f"{chip_type} ({designer})", zorder=3)
    ax.fill_between(grp['Quarter'], grp['Low'], grp['High'],
                    color=color, alpha=0.10, zorder=2)

ax.set_title('Per-Chip Total Cost Trend: Key AI Accelerators (2024–2026)', pad=14)
ax.set_xlabel('Quarter')
ax.set_ylabel('Estimated Total Chip Cost (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_billions))
ax.tick_params(axis='x', rotation=30)
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, axis='y')
plt.tight_layout()
plt.savefig('chart2_chip_trends.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


## 4  What's Driving the Cost? — Component Breakdown Over Time
Stacked area of **Logic + CoWoS packaging + HBM memory + Auxiliary** costs, aggregated across all chips.  
This reveals *which component* is responsible for the rising total.


In [ ]:
# Sum all components across all chips per quarter
stack = chip_q.groupby('Q_order', observed=True)[list(cost_cols.keys())].sum().reset_index()
stack['Quarter'] = stack['Q_order'].astype(str)
stack = stack.sort_values('Q_order')

fig, ax = plt.subplots(figsize=(13, 6))

bottom = np.zeros(len(stack))
for component, color in COMPONENT_COLORS.items():
    vals = stack[component].values
    ax.bar(stack['Quarter'], vals, bottom=bottom, color=color,
           label=component, alpha=0.88, zorder=3)
    bottom += vals

ax.set_title('AI Chip Cost Breakdown by Component (All Chips, All Designers)', pad=14)
ax.set_xlabel('Quarter')
ax.set_ylabel('Aggregate Cost (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_billions))
ax.tick_params(axis='x', rotation=30)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart3_component_breakdown.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


## 5  HBM Memory: Supply vs Cost Pressure
HBM is consistently the **single largest cost component** of AI chips.  
This chart overlays the industry-wide HBM supply (USD) against total HBM costs paid — showing scarcity-driven price pressure.


In [ ]:
supply_q['Q_order'] = pd.Categorical(supply_q['Quarter'], categories=QUARTER_ORDER, ordered=True)
supply_q = supply_q.sort_values('Q_order')
supply_q['HBM supply median'] = pd.to_numeric(supply_q['HBM supply (USD) (median)'], errors='coerce')
supply_q['HBM supply low']    = pd.to_numeric(supply_q['HBM supply (USD) (5th percentile)'], errors='coerce')
supply_q['HBM supply high']   = pd.to_numeric(supply_q['HBM supply (USD) (95th percentile)'], errors='coerce')

# Total HBM cost paid across all chips per quarter
hbm_cost = chip_q.groupby('Q_order', observed=True)['HBM'].sum().reset_index()
hbm_cost['Quarter'] = hbm_cost['Q_order'].astype(str)
hbm_cost = hbm_cost.sort_values('Q_order')

# Align on common quarters
common_q = sorted(set(supply_q['Quarter']) & set(hbm_cost['Quarter']),
                  key=lambda q: QUARTER_ORDER.index(q))

s = supply_q[supply_q['Quarter'].isin(common_q)].sort_values('Q_order')
h = hbm_cost[hbm_cost['Quarter'].isin(common_q)].sort_values('Q_order')

fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

color_supply = '#ef9a9a'
color_cost   = '#4fc3f7'

ax1.fill_between(s['Quarter'], s['HBM supply low'], s['HBM supply high'],
                 color=color_supply, alpha=0.2)
ax1.plot(s['Quarter'], s['HBM supply median'], color=color_supply,
         linewidth=2.5, marker='s', markersize=7, label='HBM Industry Supply (USD)')

ax2.fill_between(h['Quarter'],
                 chip_q[chip_q['Q_order'].isin(h['Q_order'])].groupby('Q_order', observed=True)['Total Low'].sum().values,
                 chip_q[chip_q['Q_order'].isin(h['Q_order'])].groupby('Q_order', observed=True)['Total High'].sum().values,
                 color=color_cost, alpha=0.12)
ax2.plot(h['Quarter'], h['HBM'], color=color_cost,
         linewidth=2.5, marker='o', markersize=7, label='Total HBM Cost (all chips)')

ax1.set_title('HBM Memory: Industry Supply vs Aggregate Chip Cost (2024–2025)', pad=14)
ax1.set_xlabel('Quarter')
ax1.set_ylabel('HBM Supply (USD)', color=color_supply)
ax2.set_ylabel('Total HBM Cost Paid (USD)', color=color_cost)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_billions))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_billions))
ax1.tick_params(axis='x', rotation=30)
ax1.tick_params(axis='y', colors=color_supply)
ax2.tick_params(axis='y', colors=color_cost)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)
ax1.grid(True, axis='y')
plt.tight_layout()
plt.savefig('chart4_hbm_supply_vs_cost.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()


## 6  Latest Quarter: Cost Share by Component
Which component eats the largest share of AI chip cost *right now*?


In [ ]:
latest_q = chip_q['Q_order'].max()
latest   = chip_q[chip_q['Q_order'] == latest_q]

component_totals = {c: latest[c].sum() for c in cost_cols.keys()}
total = sum(component_totals.values())

labels   = list(component_totals.keys())
sizes    = list(component_totals.values())
colors   = [COMPONENT_COLORS[l] for l in labels]
explode  = [0.05 if l == 'HBM' else 0 for l in labels]

fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(14, 6))

wedges, texts, autotexts = ax_pie.pie(
    sizes, labels=labels, colors=colors, autopct='%1.1f%%',
    startangle=140, explode=explode,
    textprops={'color': '#eeeeee', 'fontsize': 12},
    wedgeprops={'linewidth': 1.5, 'edgecolor': '#0f0f0f'}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax_pie.set_title(f'Cost Share — {latest_q}', pad=12)

# Bar chart: absolute values
bars = ax_bar.barh(labels, [s/1e9 for s in sizes], color=colors, alpha=0.88)
ax_bar.set_xlabel('Total Cost (USD Billions)')
ax_bar.set_title(f'Absolute Component Costs — {latest_q}', pad=12)
for bar, val in zip(bars, sizes):
    ax_bar.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'${val/1e9:.1f}B', va='center', fontsize=10, color='#eeeeee')
ax_bar.grid(True, axis='x')

plt.tight_layout()
plt.savefig('chart5_cost_share_latest.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f"\nLatest quarter: {latest_q}")
for k, v in component_totals.items():
    print(f"  {k:12s}  ${v/1e9:.2f}B  ({v/total*100:.1f}%)")


## 7  Key Takeaways

In [ ]:
print("=" * 62)
print("  AI Hardware Cost Rise — Key Stats (Epoch AI Dataset)")
print("=" * 62)

for designer in ['NVIDIA', 'AMD', 'Google', 'Amazon']:
    sub = design_q[design_q['Designer'] == designer].sort_values('Q_order')
    if len(sub) >= 2:
        first_q, last_q = sub.iloc[0], sub.iloc[-1]
        change_pct = (last_q['Total Cost'] - first_q['Total Cost']) / first_q['Total Cost'] * 100
        print(f"  {designer:8s}  {first_q['Quarter']} → {last_q['Quarter']}  "
              f"${first_q['Total Cost']/1e9:.1f}B → ${last_q['Total Cost']/1e9:.1f}B  "
              f"({change_pct:+.0f}%)")

print()
# HBM share trend
for qtr in ['Q1 2024', 'Q1 2025', 'Q1 2026']:
    sub = chip_q[chip_q['Quarter'] == qtr]
    if len(sub):
        total_ = sub['Total Cost'].sum()
        hbm_   = sub['HBM'].sum()
        print(f"  HBM share {qtr}: {hbm_/total_*100:.1f}%  (${hbm_/1e9:.1f}B of ${total_/1e9:.1f}B)")

print("=" * 62)
print("Source: Epoch AI — epoch.ai/data/ai-chip-components")
print("License: CC BY 4.0")
